# HR Attrition Analysis

Exploratory analysis of the IBM HR Analytics dataset. The notebook describes associations in one cross-sectional dataset; it does **not** establish causal drivers of employee attrition.

## 1. Setup

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="darkgrid")


## 2. Load and inspect the data

The notebook supports the original Kaggle path and a local `data/` fallback.

In [ ]:
KAGGLE_PATH = Path("/kaggle/input/ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv")
LOCAL_PATH = Path("data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
data_path = KAGGLE_PATH if KAGGLE_PATH.exists() else LOCAL_PATH

if not data_path.exists():
    raise FileNotFoundError(
        "Dataset not found. Place the IBM HR Analytics CSV under data/ or run this notebook on Kaggle."
    )

df = pd.read_csv(data_path)
display(df.head())
display(df.tail())


## 3. Data quality and descriptive statistics

In [ ]:
df.info()


The dataset contains no missing values in the inspected fields, so no null-imputation step is required for this analysis.

In [ ]:
df.describe()





In [ ]:
df['Attrition'].value_counts()
df['Attrition'].value_counts(normalize=True) * 100

The target is imbalanced: most employees in this dataset are in the `No` attrition class. This matters when interpreting percentages and any future predictive work.

## 4. Overtime and attrition

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="OverTime", hue="Attrition", data=df)
plt.title("Attrition by OverTime")
plt.show()


Employees recorded as working overtime show a higher attrition rate in this dataset. This is an **association**, not evidence that overtime itself causes attrition.

In [ ]:

attrition_overtime = df.groupby("OverTime")["Attrition"].value_counts(normalize=True).unstack() * 100
print(attrition_overtime.round(2))
#As expected, Workers who don't work overtime have a percent of 10.44% of resigning 
#While the ones who do overtime are expected to have a percentage of 30.53% of resigning.

## 5. Job role and attrition

In [ ]:
plt.figure(figsize=(10,5)) 
sns.countplot(x="JobRole", hue="Attrition", data=df)
plt.title("Attrition by Position")
plt.xticks(rotation=45)
plt.show()


Attrition rates differ across job roles. Raw counts and within-role rates should be considered together because role sizes differ.

In [ ]:
JobRole_Attrition= df.groupby("JobRole")["Attrition"].value_counts(normalize=True).unstack() * 100
sorted_roles = JobRole_Attrition.sort_values(by="Yes", ascending=False)
print(sorted_roles.head(3).round(2))


## 6. Income distribution and attrition

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="When grouping with a length-1 list-like")
plt.figure(figsize=(10,5))
sns.kdeplot(data=df, x="MonthlyIncome",hue="Attrition",fill=True)
plt.title("Attrition by Income")
plt.show() 

The income distributions differ between employees who stayed and left. The pattern should be treated as descriptive because income is related to other factors such as role, seniority, and tenure.

## 7. Correlation review

In [ ]:
df['Attrition_numeric'] = df['Attrition'].apply(lambda x: 1 if x == 'Yes' else 0)
numeric_df = df.select_dtypes(include=['int64', 'float64'])
corr = numeric_df.corr() 
plt.figure(figsize=(20,8))
sns.heatmap(data=corr, annot=True, fmt=".2f", cmap="coolwarm",cbar=True)
plt.title("Correlation Heatmap")
plt.show()

The correlation matrix is a descriptive screening tool. Correlation magnitude does not establish causality and can miss non-linear or conditional relationships.

In [ ]:

numeric_df = df.select_dtypes(include=['int64', 'float64'])

corr = numeric_df.corr()

if 'Attrition_numeric' in corr.columns:
    attrition_corr = corr['Attrition_numeric'].sort_values(ascending=False)
    print("Top 5 Positive Correlations:\n", attrition_corr.head(6))
    print("\nTop 5 Negative Correlations:\n", attrition_corr.tail(5))
else:
    print("Column 'Attrition_numeric' not found in numeric columns.")


            


## 8. Conclusions and limitations

The analysis identifies useful descriptive patterns across overtime, role, income, tenure, and related variables. These results should not be converted directly into HR policy or causal claims without stronger validation, confounder analysis, and appropriate fairness review.